# 06 · Train, tune and debug the CTR model
**AAI-540 · Group 4 · Criteo CTR**

First real iteration of the model: a SageMaker **XGBoost** training job on the
label-first CSVs from notebook 04, with

* **early stopping** on the validation split, so the number of rounds is
  learned rather than guessed;
* **SageMaker Debugger** rules (`LossNotDecreasing`, `Overfit`,
  `OverTraining`) watching the job, so a diverging configuration is stopped
  instead of burning an hour;
* **Automatic Model Tuning** doing a small Bayesian search over the parameters
  that actually matter for this data, optimising `validation:logloss`;
* **SageMaker Experiments** capturing every trial, so any number reported here
  can be traced to the exact job, data version and hyperparameters.

**Why XGBoost here.** The features are 13 skewed count columns and 26 hashed
categorical columns; gradient-boosted trees handle that mix without scaling and
learn interactions without us specifying them. The design document's third arm
(a DeepFM-style model with learned embeddings) is documented as a future
enhancement rather than claimed: it needs GPU instances and embedding
infrastructure the Learner Lab environment does not provide.

**What this notebook does not do:** touch the test split for any decision. The
tuner selects on validation; test is scored exactly once, at the end, for the
comparison against the notebook-05 benchmarks.

In [ ]:
import os, sys, json, time, datetime as dt
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.debugger import Rule, rule_configs
from sagemaker.estimator import Estimator
from sagemaker.image_uris import retrieve
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import ContinuousParameter, IntegerParameter, HyperparameterTuner

from criteo_ctr import config as C
from criteo_ctr import evaluation as E
from criteo_ctr import features as F
from criteo_ctr.io_utils import Store, s3_uri, canonical_columns

sess = sagemaker.Session()
boto_sess = sess.boto_session
sm = boto_sess.client("sagemaker")
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)

FIG_DIR = os.path.abspath("../reports/figures"); os.makedirs(FIG_DIR, exist_ok=True)
ART_DIR = os.path.abspath("../artifacts"); os.makedirs(ART_DIR, exist_ok=True)

XGB_PREFIX = f"{C.SPLITS_PREFIX}/xgboost"
MODEL_PREFIX = f"{C.PREFIX}/models"
image_uri = retrieve(framework="xgboost", region=region, version="1.7-1")
print(f"region={region}\nimage={image_uri}\ntrain input={s3_uri(bucket, XGB_PREFIX)}")

## 1 · Training inputs and the positive-class weight

`scale_pos_weight` is set from the **observed** training click rate rather than
left at 1.0 or copied from a tutorial. Note what the data actually says: Criteo
sub-sampled negatives before releasing this dataset, so the click rate here is
far above real-world CTR. The design document's early "heavily imbalanced"
wording was written before we measured it, and the measured rate is what the
weight is derived from.

We deliberately do **not** resample. Resampling changes the base rate, and with
it the calibration that a bidding system depends on; a weight leaves the
predicted probabilities interpretable.

In [ ]:
manifest = json.loads(boto_sess.client("s3").get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/split_manifest.json")["Body"].read())
benchmark = json.loads(boto_sess.client("s3").get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/benchmark_metrics.json")["Body"].read())

train_ctr = manifest["splits"]["train"]["ctr"]
scale_pos_weight = round((1 - train_ctr) / train_ctr, 4)

print(f"train rows        {manifest['splits']['train']['rows']:,}")
print(f"train CTR         {train_ctr:.5f}   (Criteo sub-sampled negatives: not a real-world rate)")
print(f"scale_pos_weight  {scale_pos_weight}")
print(f"benchmark gate    {benchmark['gate_threshold_log_loss']:.6f} (log loss to beat)")

inputs = {
    "train": TrainingInput(s3_data=s3_uri(bucket, f"{XGB_PREFIX}/train/"),
                           content_type="text/csv"),
    "validation": TrainingInput(s3_data=s3_uri(bucket, f"{XGB_PREFIX}/validation/"),
                                content_type="text/csv"),
}

## 2 · Baseline training job with Debugger watching

One job at the design document's starting configuration. The Debugger rules
are not decoration: `LossNotDecreasing` catches a learning rate that has
stalled the job, and `Overfit` catches the validation curve turning up while
training keeps improving — the failure this dataset invites, because a
high-cardinality categorical column lets a tree memorise individual advertisers.

In [ ]:
rules = [
    Rule.sagemaker(rule_configs.loss_not_decreasing()),
    Rule.sagemaker(rule_configs.overfit()),
    Rule.sagemaker(rule_configs.overtraining()),
]

xgb = Estimator(
    image_uri=image_uri, role=role, instance_count=1, instance_type="ml.m5.xlarge",
    output_path=s3_uri(bucket, MODEL_PREFIX), base_job_name="criteo-xgb-base",
    sagemaker_session=sess, rules=rules,
    # Metrics are regex-scraped from the container log so the training curve can
    # be plotted and so the tuner has an objective to optimise.
    metric_definitions=[
        {"Name": "validation:logloss", "Regex": r".*\[\d+\].*#011validation-logloss:(\S+)"},
        {"Name": "train:logloss", "Regex": r".*\[\d+\].*#011train-logloss:(\S+)"},
    ],
)
xgb.set_hyperparameters(
    objective="binary:logistic", eval_metric="logloss",
    max_depth=8, eta=0.1, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_lambda=1.0, num_round=500,
    early_stopping_rounds=30, scale_pos_weight=scale_pos_weight,
)

t0 = time.time()
xgb.fit(inputs, logs="None", wait=True)
base_job = xgb.latest_training_job.name
print(f"\n{base_job} finished in {time.time() - t0:.0f}s")
print(f"artifact: {xgb.model_data}")

In [ ]:
# Debugger verdicts. NoIssuesFound is the outcome we want; an IssuesFound here
# is a reason to change the configuration, not to carry on to tuning.
for summary in xgb.latest_training_job.rule_job_summary():
    print(f"{summary['RuleConfigurationName']:28s} {summary['RuleEvaluationStatus']:14s} "
          f"{summary.get('StatusDetails', '').strip()[:60]}")

description = sm.describe_training_job(TrainingJobName=base_job)
final = {m["MetricName"]: m["Value"] for m in description.get("FinalMetricDataList", [])}
print("\nfinal metrics:", {k: round(v, 5) for k, v in final.items()})
print("billable seconds:", description.get("BillableTimeInSeconds"))

## 3 · Training curve

Plotted from the metrics the job emitted, not from a local re-run. A widening
gap between the train and validation curves is the overfitting signal the
Debugger rule watches for; early stopping is what keeps the chosen round at the
bottom of the validation curve rather than the end of the schedule.

In [ ]:
import matplotlib.pyplot as plt
from sagemaker.analytics import TrainingJobAnalytics

curves = {}
for metric in ("train:logloss", "validation:logloss"):
    try:
        frame = TrainingJobAnalytics(base_job, metric_names=[metric]).dataframe()
        if not frame.empty:
            curves[metric] = frame.sort_values("timestamp")
    except Exception as exc:          # metrics can lag a few minutes after the job
        print(f"{metric}: {exc}")

if curves:
    fig, ax = plt.subplots(figsize=(7, 3.2))
    for (metric, frame), color in zip(curves.items(), ["#8aa6c1", "#2f5d8a"]):
        ax.plot(frame["timestamp"], frame["value"], label=metric, color=color)
    ax.set_xlabel("seconds into job"); ax.set_ylabel("log loss")
    ax.set_title(f"Training curve — {base_job}"); ax.legend(fontsize=8)
    plt.tight_layout()
    path = os.path.join(FIG_DIR, "14_training_curve.png")
    plt.savefig(path, dpi=140, bbox_inches="tight")
    store.upload_file(path, f"{C.PREFIX}/reports/figures/14_training_curve.png")
    plt.show()
else:
    print("no metric rows yet — re-run this cell in a few minutes")

## 4 · Automatic Model Tuning (Bayesian, on validation log loss)

Kept small on purpose: 12 jobs, 3 in parallel. The ranges cover depth
(interaction order), learning rate, row and column sampling, and
`min_child_weight`, which is the parameter that most directly limits
memorising rare hashed levels. The objective is `validation:logloss` so
selection matches both the reported metric and the pipeline gate.

A wider sweep is a Module 6 job: once the CI/CD pipeline exists, a sweep runs
without anyone sitting in a notebook waiting for it.

In [ ]:
tuner = HyperparameterTuner(
    estimator=xgb,
    objective_metric_name="validation:logloss",
    objective_type="Minimize",
    metric_definitions=xgb.metric_definitions,
    hyperparameter_ranges={
        "max_depth": IntegerParameter(4, 10),
        "eta": ContinuousParameter(0.02, 0.3, scaling_type="Logarithmic"),
        "subsample": ContinuousParameter(0.6, 1.0),
        "colsample_bytree": ContinuousParameter(0.5, 1.0),
        "min_child_weight": IntegerParameter(1, 50),
        "reg_lambda": ContinuousParameter(0.1, 10.0, scaling_type="Logarithmic"),
    },
    max_jobs=12, max_parallel_jobs=3, strategy="Bayesian",
    base_tuning_job_name="criteo-xgb-hpo",
)

t0 = time.time()
tuner.fit(inputs, include_cls_metadata=False, wait=True)
print(f"tuning finished in {time.time() - t0:.0f}s: {tuner.latest_tuning_job.name}")

In [ ]:
from sagemaker.analytics import HyperparameterTuningJobAnalytics

trials = HyperparameterTuningJobAnalytics(
    tuner.latest_tuning_job.name, sagemaker_session=sess).dataframe()
cols = [c for c in ["FinalObjectiveValue", "max_depth", "eta", "subsample",
                    "colsample_bytree", "min_child_weight", "reg_lambda",
                    "TrainingJobName", "TrainingJobStatus"] if c in trials.columns]
trials = trials.sort_values("FinalObjectiveValue")
display(trials[cols].head(12).round(5))

best_job = tuner.best_training_job()
best = sm.describe_training_job(TrainingJobName=best_job)
best_params = {k: v for k, v in best["HyperParameters"].items()
               if not k.startswith("_") and k not in ("objective", "eval_metric")}
print(f"\nbest job: {best_job}")
print(json.dumps(best_params, indent=2))

In [ ]:
# Which parameter actually moved the objective. Useful for the next sweep, and
# honest about how much of the improvement came from tuning at all.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
for ax, param in zip(axes, ["max_depth", "eta", "min_child_weight"]):
    if param in trials.columns:
        ax.scatter(trials[param], trials["FinalObjectiveValue"], color="#6a8caf")
        ax.set_xlabel(param); ax.set_ylabel("validation log loss")
        if param == "eta":
            ax.set_xscale("log")
axes[0].set_title("Tuning: objective against each parameter")
plt.tight_layout()
path = os.path.join(FIG_DIR, "15_tuning.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/15_tuning.png")
plt.show()

spread = trials["FinalObjectiveValue"]
print(f"objective across {len(trials)} trials: best {spread.min():.5f}, "
      f"worst {spread.max():.5f}, spread {spread.max() - spread.min():.5f}")

## 5 · Score the tuned model on the held-out test split — once

The artefact is downloaded and scored locally rather than through an endpoint,
because at this point there is no endpoint and the question is only "what does
this model score". The deployed paths are built and parity-checked in notebook 07.

`scripts/evaluate.py` is the same code the pipeline's evaluation step runs, so
the number the gate will see in notebook 10 is produced by the same function.

In [ ]:
import subprocess, tarfile, shutil

work = os.path.abspath("../artifacts/model_eval"); shutil.rmtree(work, ignore_errors=True)
os.makedirs(f"{work}/model", exist_ok=True); os.makedirs(f"{work}/test", exist_ok=True)

best_artifact = best["ModelArtifacts"]["S3ModelArtifacts"]
sagemaker.s3.S3Downloader.download(best_artifact, f"{work}/model", sagemaker_session=sess)
sagemaker.s3.S3Downloader.download(s3_uri(bucket, f"{XGB_PREFIX}/test/"),
                                   f"{work}/test", sagemaker_session=sess)
print("downloaded:", os.listdir(f"{work}/model"), os.listdir(f"{work}/test"))

result = subprocess.run(
    [sys.executable, "../scripts/evaluate.py",
     "--model-dir", f"{work}/model", "--test-dir", f"{work}/test",
     "--output-dir", f"{work}/evaluation", "--model-name", "criteo-xgboost",
     "--benchmark-compare"],
    capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    raise RuntimeError(result.stderr[-2000:])

evaluation = json.load(open(f"{work}/evaluation/evaluation.json"))
metrics = evaluation["binary_classification_metrics"]

## 6 · Compare against the notebook-05 benchmarks

In [ ]:
rows = []
for name, report in benchmark["reports"].items():
    rows.append({"model": name, "log_loss": report["log_loss"],
                 "normalised_entropy": report["normalised_entropy"],
                 "roc_auc": report["roc_auc"], "pr_auc": report["pr_auc"],
                 "calibration_error": report["expected_calibration_error"]})
rows.append({"model": "criteo-xgboost (tuned)",
             "log_loss": metrics["log_loss"]["value"],
             "normalised_entropy": metrics["normalised_entropy"]["value"],
             "roc_auc": metrics["auc"]["value"], "pr_auc": metrics["pr_auc"]["value"],
             "calibration_error": metrics["expected_calibration_error"]["value"]})
table = pd.DataFrame(rows).set_index("model")
display(table.round(5))

gate = benchmark["gate_threshold_log_loss"]
candidate = metrics["log_loss"]["value"]
improvement = (gate - candidate) / gate
print(f"\ncandidate test log loss   {candidate:.6f}")
print(f"benchmark (gate) log loss {gate:.6f}")
print(f"improvement               {improvement:.2%}")
print(f"normalised entropy        {metrics['normalised_entropy']['value']:.4f} "
      f"({'beats' if metrics['normalised_entropy']['value'] < 1 else 'does NOT beat'} the base rate)")
print(f"\nquality gate would {'PASS' if candidate <= gate else 'FAIL'} at this threshold.")

In [ ]:
# Calibration of the tuned model, and feature importance — the two things a
# stakeholder asks about after "is it better".
calibration = pd.DataFrame(evaluation["evaluation_context"]["calibration"])
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))

axes[0].plot(calibration["mean_predicted"], calibration["observed_ctr"], "o-",
             color="#2f5d8a", label="criteo-xgboost")
lim = [0, calibration["mean_predicted"].max() * 1.15]
axes[0].plot(lim, lim, ls="--", lw=1, color="#555", label="perfect calibration")
axes[0].set_xlabel("mean predicted"); axes[0].set_ylabel("observed CTR")
axes[0].set_title("Calibration — tuned model"); axes[0].legend(fontsize=8)

try:
    import xgboost as xgb_local
    booster = xgb_local.Booster()
    tar_path = [p for p in os.listdir(f"{work}/model") if p.endswith(".tar.gz")]
    if tar_path:
        with tarfile.open(f"{work}/model/{tar_path[0]}") as tar:
            tar.extractall(f"{work}/model", filter="data")
    booster.load_model([f"{work}/model/{p}" for p in os.listdir(f"{work}/model")
                        if not p.endswith(".tar.gz")][0])
    gains = booster.get_score(importance_type="gain")
    cols = F.model_feature_columns()
    named = {cols[int(k[1:])] if k.startswith("f") and k[1:].isdigit() else k: v
             for k, v in gains.items()}
    top = pd.Series(named).sort_values(ascending=False).head(15)[::-1]
    axes[1].barh(top.index, top.values, color="#6a8caf")
    axes[1].set_title("Top 15 features by gain")
    axes[1].set_xlabel("gain")
except Exception as exc:
    axes[1].text(0.5, 0.5, f"importance unavailable:\n{exc}", ha="center", va="center")

plt.tight_layout()
path = os.path.join(FIG_DIR, "16_model_evaluation.png")
plt.savefig(path, dpi=140, bbox_inches="tight")
store.upload_file(path, f"{C.PREFIX}/reports/figures/16_model_evaluation.png")
plt.show()

Feature importance is reported with a caveat that belongs in the design
document too: the categorical columns are **hashed**, so "C14 carries most of
the gain" tells us a high-cardinality identifier matters, not *what* it
identifies. This is the auditability gap the security section names — we can
see which features drive predictions and still cannot rule out that one of them
proxies for a protected attribute.

## 7 · Publish the model record

In [ ]:
record = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "base_job": base_job,
    "tuning_job": tuner.latest_tuning_job.name,
    "best_training_job": best_job,
    "best_hyperparameters": best_params,
    "model_artifact": best_artifact,
    "image_uri": image_uri,
    "scale_pos_weight": scale_pos_weight,
    "train_rows": manifest["splits"]["train"]["rows"],
    "test_rows": manifest["splits"]["test"]["rows"],
    "evaluation": evaluation,
    "benchmark_gate_log_loss": gate,
    "improvement_vs_benchmark": improvement,
    "debugger_rules": [s["RuleConfigurationName"] for s in
                       xgb.latest_training_job.rule_job_summary()],
    "tuning_trials": int(len(trials)),
}
store.put_json(record, f"{C.ARTIFACTS_PREFIX}/model_record.json")
store.upload_file(f"{work}/evaluation/evaluation.json",
                  f"{MODEL_PREFIX}/evaluation/evaluation.json")
with open(os.path.join(ART_DIR, "model_record.json"), "w") as fh:
    json.dump(record, fh, indent=2, default=str)

print(f"wrote s3://{bucket}/{C.ARTIFACTS_PREFIX}/model_record.json")
print(f"evaluation.json -> s3://{bucket}/{MODEL_PREFIX}/evaluation/evaluation.json")
print("\nNext: 07_deployment.ipynb registers this artefact, deploys it two ways,\n"
      "and checks that both paths return the same scores.")